In [116]:
import pandas as pd
import numpy as np

_Создадим матрицу, для обучения._

In [117]:
df = np.array(pd.read_csv('mt_train.csv', sep=';', header=None))
df

array([[  67,   42,   39, 2043],
       [  35,   94,   46, 2695],
       [  31,   36,   55, 2005],
       [  80,   50,   48, 2478],
       [  41,   58,   59, 2491],
       [  44,   80,   98, 3701],
       [  43,   39,   37, 1752],
       [  93,   91,   52, 3287],
       [  87,   31,   30, 1822],
       [  90,   38,   43, 2245],
       [  77,   60,   33, 2228],
       [  51,   79,   87, 3491],
       [  33,   98,   54, 2906],
       [  73,   56,   82, 3240],
       [  71,   45,   94, 3319],
       [  98,   55,   37, 2424],
       [  56,   55,   52, 2407],
       [  39,   97,   53, 2945],
       [  57,   67,   87, 3422],
       [  68,   38,   62, 2475],
       [  64,   40,   53, 2271],
       [  45,   55,   92, 3215],
       [  76,   62,   53, 2753],
       [  85,   95,   33, 2824],
       [  30,   36,   82, 2547]])

_Определим метки и таргеты._

In [118]:
X_train = df[:, :3]
Y = df[:, 3]

#### **Стандартизация данных.**

_Сначала вычислим значения, которые не будут меняться. Это среднее значение._

In [119]:
train_mean = X_train.mean(axis=0)

_И среднее стандартное отклонение._ **train_std**

In [120]:
#найдем отклонение от среднего на каждом объекте
train_deviation = (X_train - train_mean) ** 2 #матрица
#дисперсия, среднее квадратов отклонения
train_variance = train_deviation.mean(axis=0) #вектор
#стандартное отклонение для каждого признака
train_std = np.sqrt(train_variance)

_Теперь напишем функцию для стандартизации любых поступающих данных._

In [121]:
def stand(data):
    #стандартизируем признаки
    data_stand = (data - train_mean) / train_std
    return data_stand

_Стандартизируем признаки и сразу добавим свободный коэффициент._

train_mean = X_train.mean(axis=0)

train_std = X_train.std(axis=0)

X_train = (X_train - train_mean) / train_std

X_test = (X_test - train_mean) / train_std

In [122]:
X_train = np.c_[stand(X_train), np.ones(X_train.shape[0])]

#### **Полно пакетный градиентный спуск.**

In [123]:
def batch_gradient_descent(X, Y, w, lr):
    #предсказание на данный момент
    pred = X @ w.T #вектор
    #ошибка на каждом объекте
    error = (pred - Y)[:, None] #вектор
    #средний градиент для веса
    grad = 2 * (X * error).mean(axis=0) #вектор
    #изменим веса
    w -= grad * lr
    return w

#### **Среднеквадратическая ошибка.**

In [124]:
def rmse(Y_pr, Y):
    #ошибка на каждом объекте
    error = (Y_pr - Y) ** 2 #вектор
    #средняя ошибка
    error_mean = error.mean() #скаляр
    #корень из средней ошибки
    error_rmse = np.sqrt(error_mean) #скаляр
    return error_rmse

#### **Обучение.**
_Проводим обучение до тех пор, пока ошибка меняется больше, чем на 0.001_

In [125]:
lr = 0.01
w = np.zeros(X_train.shape[1]) #начальные нулевые веса
rmse_last = float('inf')
for _ in range(100000):
    #вычисляем текущую ошибку
    cur_rmse = rmse(X_train @ w.T, Y)
    #проверяем насколько изменилась ошибка с прошлого раза
    if rmst_last - cur_rmse < 0.001:
        break
    else:
        #если изменилась недостаточно передаем last
        rmse_last = cur_rmse
    #корректируем веса
    w = batch_gradient_descent(X_train, Y, w, lr)
print(rmse(X_train @ w.T, Y), w)

17.910609999291367 [ 175.59127402  323.3512715   460.19629398 2679.44      ]


#### **Проверка на других тестовыех данных.**

_Считываем данные._

In [126]:
df_test = np.array(pd.read_csv('mt_test.csv', sep=';', header=None))
df_test

array([[  32,   51,   37, 1848],
       [  73,   50,   60, 2654],
       [  66,   37,   75, 2736],
       [  98,   87,   43, 3053],
       [  40,   53,   37, 1886],
       [  54,   50,   62, 2562],
       [  42,   95,   90, 3764],
       [  54,   32,   40, 1787],
       [  84,   96,   78, 3825],
       [  84,   45,   35, 2145],
       [  47,   72,   50, 2527],
       [  78,   52,   43, 2363],
       [  83,   40,   85, 3118],
       [  91,   86,   51, 3138],
       [  55,   44,   43, 2042],
       [  73,   36,   86, 3005],
       [  89,   45,   54, 2585],
       [  39,   96,   83, 3585],
       [  99,   66,   51, 2948],
       [  70,   79,   77, 3435],
       [  70,   75,   46, 2702],
       [  58,   75,   97, 3725],
       [  96,   76,   30, 2564],
       [  59,   93,   65, 3301],
       [  83,   63,   32, 2316]])

_Стандартизируем их в соответствии со средним отклонение и средним значением, которые были получены на трейне._

In [127]:
X_test_stand = stand(df_test[:, :3])

_Определяем признаки и таргеты._

In [128]:
X_test = np.c_[X_test_stand, np.ones(df_test.shape[0])]
Y_test = df_test[:, 3]

_Получаем среднюю ошибку на тестовых данных._

In [129]:
print(rmse(X_test @ w.T, Y_test))

17.30776994678604
